# 10 · ML · Evolução do baseline: valor do pedido

**Squad 2 · Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**
**Problema P3:** estimar o `valor_total` do pedido a partir do contexto da compra.

O baseline do notebook 08 (v1) acerta bem o histórico (R² 0,82), mas no tempo real, que é onde o modelo vai
pontuar, o R² cai para 0,08. Este notebook aplica as melhorias uma de cada vez, mede cada uma no **mesmo teste
do tempo real** (os 367 pedidos mais recentes do stream) e grava a melhor versão no lake.

Técnica: LightGBM. No notebook 09 ele empata com o CatBoost no erro, nesta mesma configuração, e treina mais
de 10 vezes mais rápido.

| Entrada | Saída |
|---|---|
| `squad2/grupo2/ml/features_valor_pedido` | `squad2/grupo2/ml/modelo_valor_pedido` (nova versão, append) |

In [ ]:
%pip install -q lightgbm==4.7.0

In [ ]:
%run ./00_setup_config

In [ ]:
from pyspark.sql import functions as F
import lightgbm as lgb, numpy as np, pandas as pd, matplotlib.pyplot as plt, json, datetime as dt
from sklearn.metrics import mean_absolute_error, r2_score

CAMINHO_FEATURES, CAMINHO_MODELO = f"{BASE}/ml/features_valor_pedido", f"{BASE}/ml/modelo_valor_pedido"
CATEGORICAS = ["categoria_dominante_idx", "metodo_pagamento_idx", "periodo_idx", "uf_idx"]
FEATURES = ["hora", "dia_semana", "fim_de_semana", "n_itens", "qtd_total", "n_categorias", "qtd_por_item", "uf_informada"] + CATEGORICAS
FEATURES_V3 = FEATURES + ["eh_stream"]
BASE_LBL, C_LBL, CENARIO_V4 = "baseline (média por categoria, stream)", "c: só stream (08)", "v4 hist+stream sem fora da curva, teste stream"
PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.9, random_state=42, verbose=-1)
AZUL, AZUL_CLARO, CINZA = "#1F4E8C", "#5B8FC9", "#8C8C8C"
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.titleweight": "bold", "axes.edgecolor": "#595959", "axes.grid": True, "grid.alpha": .3})

fs = ler_delta(CAMINHO_FEATURES)
if fs is None: raise FileNotFoundError("Rode o notebook 07 antes: tabela de features nao encontrada")
df = fs.select("fonte", "conjunto", "id_pedido", "dt_local", "categoria_dominante", "flag_item_fora_curva", "valor_total", *FEATURES).toPandas()
df[CATEGORICAS] = df[CATEGORICAS].astype("int32")
df["eh_stream"] = (df.fonte == "stream").astype("int32")
H, H_teste = df[(df.fonte == "historico") & (df.conjunto == "treino")], df[(df.fonte == "historico") & (df.conjunto == "teste")]
S, S_teste = df[(df.fonte == "stream") & (df.conjunto == "treino")], df[(df.fonte == "stream") & (df.conjunto == "teste")]
print(f"treino: histórico {len(H):,} | stream {len(S):,}   teste: histórico {len(H_teste):,} | stream {len(S_teste):,}")

## Diagnóstico do v1: por que o tempo real erra

Duas causas, as duas medidas no 07/08:

1. **Nível diferente.** O pedido do tempo real tem bem menos itens e unidades que o do histórico; o modelo
   treinado só no histórico nunca viu esse perfil de carrinho e erra o nível do ticket.
2. **Itens com preço fora da curva** (preço ≥ 5× o de lista, 5% dos pedidos do stream). Nenhuma variável de
   contexto prevê esse preço, e cada pedido desses gera um erro de centenas ou milhares de reais.

Por isso cada versão é medida de duas formas: em **todos** os pedidos do teste e nos pedidos **sem item fora da
curva**, que é o comportamento normal da loja. O teste é sempre o mesmo.

In [ ]:
perfil = df[df.conjunto == "treino"].groupby("fonte").agg(pedidos=("id_pedido", "size"), ticket=("valor_total", "mean"),
         itens=("n_itens", "mean"), unidades=("qtd_total", "mean"), pct_fora_curva=("flag_item_fora_curva", "mean"))
perfil["ticket_por_unidade"] = (df[df.conjunto == "treino"].assign(r=lambda d: d.valor_total / d.qtd_total).groupby("fonte").r.median())
display(perfil.round(3).reset_index())

def erro_quadratico_fora(y, p, flag):
    e = (np.asarray(y) - np.asarray(p)) ** 2; return e[np.asarray(flag) == 1].sum() / e.sum()

## As versões

| Versão | O que muda | Por quê |
|---|---|---|
| v1 | baseline do 08: LightGBM treinado só no histórico | ponto de partida |
| v2 | v1 + correção de nível: soma à previsão (em log) o erro médio do v1 no treino do stream | ataca a causa 1 sem retreinar |
| v3 | treino conjunto histórico + stream, com a coluna `eh_stream` | o modelo aprende o nível de cada fonte e usa o volume do histórico |
| v4 | v3 sem os pedidos com item fora da curva no treino | ataca a causa 2: o modelo deixa de aprender ruído de preço |

Duas referências ficam ao lado: o **baseline** (média do ticket por categoria no treino do stream, o mesmo do
cenário c do 08) e o **modelo c do 08** (LightGBM treinado só no stream). Ganhar do v1 é pouco; o v4 tem que
ganhar também dessas duas.

Hiperparâmetros iguais em todas as versões (os do 08). Testamos variações de folhas, taxa e função de perda: o
MAE muda menos de R$ 2, então não entram, e escolher por elas seria ajustar o modelo ao próprio teste.

In [ ]:
def treinar(tr, feats):
    return lgb.LGBMRegressor(**PARAMS).fit(tr[feats], np.log1p(tr.valor_total), categorical_feature=CATEGORICAS)

v1 = treinar(H, FEATURES)
ajuste_nivel = float((np.log1p(S.valor_total) - v1.predict(S[FEATURES])).mean())
treino_v3 = pd.concat([H, S]); treino_v4 = treino_v3[treino_v3.flag_item_fora_curva == 0]
v3, v4, c = treinar(treino_v3, FEATURES_V3), treinar(treino_v4, FEATURES_V3), treinar(S, FEATURES)
media_cat = lambda tr, d: d.categoria_dominante.map(tr.groupby("categoria_dominante").valor_total.mean()).fillna(tr.valor_total.mean()).values
prever = {BASE_LBL: lambda d: media_cat(S, d), C_LBL: lambda d: np.expm1(c.predict(d[FEATURES])),
          "v1": lambda d: np.expm1(v1.predict(d[FEATURES])), "v2": lambda d: np.expm1(v1.predict(d[FEATURES]) + ajuste_nivel),
          "v3": lambda d: np.expm1(v3.predict(d[FEATURES_V3])), "v4": lambda d: np.expm1(v4.predict(d[FEATURES_V3]))}
P = {v: f(S_teste) for v, f in prever.items()}
limpo, y = (S_teste.flag_item_fora_curva == 0).values, S_teste.valor_total.values

evolucao = pd.DataFrame([dict(versao=v, mae=mean_absolute_error(y, p), r2=r2_score(y, p), mae_limpo=mean_absolute_error(y[limpo], p[limpo]),
                              r2_limpo=r2_score(y[limpo], p[limpo]), pct_erro_ate_50=np.mean(np.abs(p - y) <= 50) * 100,
                              r2_historico=r2_score(H_teste.valor_total, prever[v](H_teste)) if v in ("v1", "v3", "v4") else np.nan)
                         for v, p in P.items()]).round(3)
ev = evolucao.set_index("versao")
print(f"ajuste de nível do v2 (log): {ajuste_nivel:+.3f} = previsão x {np.exp(ajuste_nivel):.2f}")
display(evolucao)

## A melhora é real?

Bootstrap pareado nos 367 pedidos do teste (2.000 sorteios com reposição): intervalo de 95% da queda do MAE do
v4 em relação a cada referência (baseline, modelo c do 08, v1, v2 e v3). Se o intervalo inteiro fica acima de
zero, a melhora não é acaso da amostra.

A versão gravada é a melhor das quatro neste mesmo teste. Para saber se essa escolha depende do teste, a seção
seguinte repete a comparação numa validação separada, sem tocar no teste.

In [ ]:
rng = np.random.default_rng(42)
erro_v4, linhas = np.abs(P["v4"] - y), []
for ref in [BASE_LBL, C_LBL, "v1", "v2", "v3"]:
    erro_ref = np.abs(P[ref] - y)
    for rotulo, m in [("todos", np.ones(len(y), bool)), ("sem fora da curva", limpo)]:
        d = erro_ref[m] - erro_v4[m]; queda = d[rng.integers(0, len(d), (2000, len(d)))].mean(1)
        linhas.append(dict(v4_contra=ref, pedidos=rotulo, queda_mae=d.mean(), ic95_inf=np.percentile(queda, 2.5),
                           ic95_sup=np.percentile(queda, 97.5), queda_pct=100 * d.mean() / erro_ref[m].mean()))
display(pd.DataFrame(linhas).round(2))
print(f"v4: os {(~limpo).sum()} pedidos com item fora da curva ({(~limpo).mean():.1%} do teste) concentram "
      f"{erro_quadratico_fora(y, P['v4'], ~limpo):.0%} do erro quadrático, e é isso que segura o R² de todos os pedidos")

## Conferências: escolha sem olhar o teste e viés da previsão

1. **Validação separada.** Os 20% mais recentes do treino do stream viram validação; as versões são retreinadas
   sem eles e comparadas ali. Se o v4 também ganhar, a escolha não dependeu do teste.
2. **Viés.** O treino em log mira o valor típico, não a média. Comparar a soma das previsões com a soma real
   mostra se o modelo serve para projetar receita somando pedidos.
3. **Período do teste.** Em que dias caem os pedidos do teste do tempo real.

In [ ]:
S_ord = S.sort_values("dt_local"); corte = int(len(S_ord) * .8); S_tr, S_val = S_ord.iloc[:corte], S_ord.iloc[corte:]
v1_val = treinar(H, FEATURES); aj_val = float((np.log1p(S_tr.valor_total) - v1_val.predict(S_tr[FEATURES])).mean())
base_val = pd.concat([H, S_tr])
cand = {"c": treinar(S_tr, FEATURES), "v3": treinar(base_val, FEATURES_V3), "v4": treinar(base_val[base_val.flag_item_fora_curva == 0], FEATURES_V3)}
prev_val = {"v1": np.expm1(v1_val.predict(S_val[FEATURES])), "v2": np.expm1(v1_val.predict(S_val[FEATURES]) + aj_val),
            "c": np.expm1(cand["c"].predict(S_val[FEATURES])), "v3": np.expm1(cand["v3"].predict(S_val[FEATURES_V3])),
            "v4": np.expm1(cand["v4"].predict(S_val[FEATURES_V3]))}
display(pd.DataFrame({"mae_validacao": {v: mean_absolute_error(S_val.valor_total, p) for v, p in prev_val.items()}}).round(1).sort_values("mae_validacao"))

vies = pd.DataFrame([dict(pedidos=r, soma_prevista=P["v4"][m].sum(), soma_real=y[m].sum(), dif_pct=100 * (P["v4"][m].sum() / y[m].sum() - 1),
                          vies_medio=(P["v4"][m] - y[m]).mean()) for r, m in [("todos", np.ones(len(y), bool)), ("sem fora da curva", limpo)]])
display(vies.round(1))
print("pedidos do teste do tempo real por dia:", {str(k): v for k, v in S_teste.groupby(S_teste.dt_local.dt.date).size().items()})

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))
vs = [C_LBL, "v1", "v2", "v3", "v4"]; x = np.arange(len(vs))
for i, (col, tit) in enumerate([("mae", "MAE, todos os pedidos (R$)"), ("mae_limpo", "MAE, sem item fora da curva (R$)"),
                                ("r2", "R², todos os pedidos"), ("r2_limpo", "R², sem item fora da curva")]):
    b = ax[i].bar(x, ev.loc[vs, col], color=[CINZA] + [AZUL_CLARO] * 3 + [AZUL]); ax[i].bar_label(b, fmt="%.2f" if "r2" in col else "%.0f", fontsize=9)
    ax[i].axhline(ev.loc[BASE_LBL, col], color=CINZA, ls="--", lw=1.2, label=BASE_LBL)
    ax[i].set_xticks(x, ["c (08)"] + vs[1:]); ax[i].set_title(tit)
fig.legend(*ax[0].get_legend_handles_labels(), loc="lower center", ncol=1, fontsize=9, frameon=False)
plt.suptitle("Evolução no teste do tempo real (367 pedidos)", fontweight="bold"); plt.tight_layout(rect=(0, .06, 1, 1)); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for v, cor in [("v1", CINZA), ("v4", AZUL)]:
    e = np.sort(np.abs(P[v][limpo] - y[limpo])); ax.plot(e, np.arange(1, len(e) + 1) / len(e) * 100, color=cor, lw=2, label=v)
ax.set_xlim(0, 400); ax.set_xlabel("erro absoluto (R$)"); ax.set_ylabel("% dos pedidos com erro até x")
ax.set_title("Distribuição do erro, pedidos sem item fora da curva"); ax.legend(); plt.tight_layout(); plt.show()

## Gravar o v4 como nova versão do modelo

Mesma tabela e mesmo formato do 08 (booster como texto). Duas diferenças para quem for pontuar:

- A lista de features inclui `eh_stream`, que o 07 não grava: o micro-lote do tempo real recebe `eh_stream = 1`
  antes de chamar o modelo (`lote.assign(eh_stream=1)[cols]`).
- No 08, `mae` e `r2` são do teste do histórico; aqui são do teste do tempo real, e o `cenario` diz isso. Para
  escolher a versão, filtrar pelo `cenario`, não comparar `r2` entre linhas de cenários diferentes.

In [ ]:
anterior = ler_delta(CAMINHO_MODELO)
versao = 1 if anterior is None else anterior.agg(F.max("versao")).first()[0] + 1
m4 = ev.loc["v4"]
registro = spark.createDataFrame([(versao, dt.datetime.now(), CENARIO_V4, float(m4.mae), float(m4.r2),
            float(ev.loc[BASE_LBL, "mae"]), json.dumps(FEATURES_V3), json.dumps(PARAMS), evolucao.to_json(orient="records", force_ascii=False),
            v4.booster_.model_to_string())],
            "versao int, dt_treino timestamp, cenario string, mae double, r2 double, mae_baseline double, features string, params string, metricas_cenarios string, modelo string")
registro.write.format("delta").options(**adls_options).mode("append").save(CAMINHO_MODELO)
print(f"[ok] versao {versao} gravada em {CAMINHO_MODELO.split('.net/')[-1]}")

In [ ]:
ultima = ler_delta(CAMINHO_MODELO).filter(F.col("cenario") == CENARIO_V4).orderBy(F.desc("versao")).first()
booster, cols = lgb.Booster(model_str=ultima.modelo), json.loads(ultima.features)
lote = S_teste.drop(columns="eh_stream").sample(5, random_state=7)
pontuado = lote[["id_pedido", "categoria_dominante", "n_itens", "qtd_total", "valor_total"]].assign(
    previsto_v1=prever["v1"](lote).round(2), previsto_v4=np.expm1(booster.predict(lote.assign(eh_stream=1)[cols])).round(2))
assert np.allclose(pontuado.previsto_v4, prever["v4"](lote.assign(eh_stream=1)).round(2)), "modelo relido difere do treinado"
print(f"versao {ultima.versao} ({ultima.cenario}, treinado em {ultima.dt_treino:%d/%m/%Y %H:%M})")
display(pontuado)

## Resumo

Números da execução de teste, todos no mesmo teste do tempo real (367 pedidos mais recentes do stream):

| Versão | MAE todos | MAE sem fora da curva | R² todos | R² sem fora da curva | Pedidos com erro ≤ R$ 50 | R² no teste do histórico |
|---|---|---|---|---|---|---|
| baseline (média por categoria, stream) | R$ 191 | R$ 148 | 0,00 | −0,17 | 23% | |
| c: só stream (08) | R$ 137 | R$ 90 | 0,04 | 0,41 | 46% | |
| v1: só histórico (08) | R$ 149 | R$ 106 | 0,08 | 0,34 | 37% | 0,82 |
| v2: v1 + correção de nível | R$ 139 | R$ 95 | 0,06 | 0,37 | 43% | |
| v3: histórico + stream | R$ 126 | R$ 80 | 0,06 | 0,48 | 51% | 0,82 |
| **v4: v3 sem fora da curva no treino** | **R$ 123** | **R$ 77** | 0,06 | **0,49** | **57%** | 0,82 |

- **O v4 é a melhor versão e a melhora é real.** O erro médio cai de R$ 149 (v1) para R$ 123, 18% a menos, com
  intervalo de 95% de R$ 19 a R$ 34 de queda. Contra o baseline a queda é de 36% (R$ 68), e contra o modelo
  treinado só no stream (c do 08), de 10% (R$ 14, intervalo de R$ 9 a R$ 19). Cada versão erra menos que a
  anterior, e o v4 ganha de todas as referências com intervalo de 95% acima de zero. Na validação separada, sem
  olhar o teste, o v4 também é o melhor (MAE R$ 103, contra R$ 108 do v3 e R$ 113 do c).
- **Nos pedidos normais a precisão sobe bastante.** Sem os pedidos com item fora da curva, o R² vai de 0,34 para
  0,49 e o erro médio de R$ 106 para R$ 77. Os pedidos estimados com erro de até R$ 50 passam de 37% para 57%.
- **O que mais ajudou foi treinar com as duas fontes juntas (v3).** A coluna `eh_stream` deixa o modelo aprender
  o nível do tempo real sem perder o volume do histórico. A correção de nível sozinha (v2) ajuda menos, e tirar
  os pedidos fora da curva do treino (v4) acrescenta pouco, mas com ganho confirmado (R$ 3, intervalo de R$ 1 a R$ 5).
- **O histórico não piorou.** O R² no teste do histórico continua 0,82 no v3 e no v4 (esperado, já que o histórico
  é 98% do treino).
- **O R² de todos os pedidos não sobe.** Os 15 pedidos com item fora da curva (4,1% do
  teste) concentram 92% do erro quadrático do v4. O preço desses itens é 5 vezes ou mais o de lista e nenhuma
  variável de contexto prevê isso. O R² geral fica entre 0,06 e 0,08 em todas as versões, e o do v1 (0,08) é até
  um pouco maior, porque ele prevê valores mais altos e chega mais perto desses pedidos. Por isso a evolução é
  medida pelo MAE e pelo R² dos pedidos normais. Se esses preços forem erro de cadastro, o tratamento é na
  qualidade de dado (a regra T4 da Silver de itens só barra preço baixo); se forem reais, nenhuma variável
  disponível os prevê.
- **Limites.** O teste do tempo real cobre praticamente um dia (346 dos 367 pedidos são de 08/10). O modelo
  prevê abaixo do valor real: a soma das previsões fica 25% abaixo da receita do teste (10% nos pedidos normais),
  porque o treino em log mira o valor típico. Ele serve para estimar o pedido típico, não para projetar receita
  somando previsões.
- **Modelo gravado.** O v4 entrou como versão 2 em `squad2/grupo2/ml/modelo_valor_pedido`. Relido da tabela,
  pontuou 5 pedidos do tempo real (com `eh_stream = 1`) com a mesma previsão do modelo em memória.